# Create configs and execute open_icu pipeline

In [1]:
from pathlib import Path

import json
import polars as pl

## Sources

In [2]:
# ricu concept dict
ricu_cnpt_dict_df = pl.read_json("concept-dict.json")
# miiv data
miiv_d_items_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
miiv_d_labitems_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

# Function
creates two dicts with usable and (currently) unusable concept informations and prepares the usable information

In [ ]:
def ricu_cnpt_dict_to_cust_cnpt_dict(info: pl.DataFrame, src_name: str, src_items: dict[str, pl.LazyFrame], src_item_col_name: str) -> tuple[dict, dict]:
    data: dict[str, dict] = {}
    un_src_data: dict = {}
    i = 0
    for cnpt in info.columns:
        if(cnpt_srcs := info[cnpt][0].get("sources")) and cnpt_srcs.get(src_name):
            for i in range(len(cnpt_srcs[src_name])):
                src = cnpt_srcs[src_name][i]
                cnpt_ids = src.get("ids")
                cnpt_regex = src.get("regex")
                if cnpt_ids is None and cnpt_regex is None:
                    un_src_data[cnpt] = info[cnpt]
                    continue
                cnpt_entr = info[cnpt][0]
                # cnpt_cat = cnpt_entr["category"].replace(" ", "_").replace("/", "_per_")
                cnpt_name = cnpt_entr["description"].replace(" ", "_").replace("/", "_per_")
                cnpt_tbl = src["table"]

                if not data.get(cnpt_cat):
                    data[cnpt_cat] = {}

                if isinstance(cnpt_ids, str) or (isinstance(cnpt_ids, list) and isinstance(cnpt_ids[0], str)):
                    un_src_data[cnpt] = info[cnpt]
                    continue
                
                if cnpt_ids is not None:
                    cnpt_ids: list[int] = cnpt_ids if isinstance(cnpt_ids, list) else [cnpt_ids]

                if data[cnpt_cat].get(cnpt_name) is None:
                    data[cnpt_cat][cnpt_name] = []
                
                data[cnpt_cat][cnpt_name].append({
                    "name": cnpt_name,
                    "unit": cnpt_entr.get("unit"),
                    "table": cnpt_tbl,
                    "code": None,
                    "ids": cnpt_ids,
                    "src_regex": cnpt_regex,
                    "short": cnpt,
                    "min": cnpt_entr.get("min"),
                    "max": cnpt_entr.get("max"),
                })
                if cnpt_ids is not None:
                    filtered_src_items = src_items[cnpt_tbl if cnpt_tbl in src_items.keys() else "else"].filter(pl.col(src_item_col_name).is_in(cnpt_ids)).select("code").collect()
                    regex = "|".join(list(filtered_src_items["code"]))
                else:
                    regex = cnpt_regex

                data[cnpt_cat][cnpt_name][i]["code"] = "(" + regex + ")"

                if data[cnpt_cat][cnpt_name][i]["max"] is None:
                    data[cnpt_cat][cnpt_name][i]["max"] = "Inf"

                if data[cnpt_cat][cnpt_name][i]["min"] is None:
                    data[cnpt_cat][cnpt_name][i]["min"] = "-Inf"

    return (data, un_src_data)

## Execution

In [4]:
data_by_cat = {}
un_src_data = {}

src_item_mapping = {"labevents": miiv_d_labitems_lf, "else": miiv_d_items_lf}
(data_by_cat, un_src_data) = ricu_cnpt_dict_to_cust_cnpt_dict(ricu_cnpt_dict_df, "miiv", src_item_mapping, "itemid")

In [21]:
data_by_cat

{'hematology': {'basophils': [{'name': 'basophils',
    'unit': '%',
    'table': 'labevents',
    'code': '(51146//Basophils)',
    'ids': [51146],
    'src_regex': None,
    'short': 'basos',
    'min': 0,
    'max': 50}],
  'mean_corpuscular_volume': [{'name': 'mean_corpuscular_volume',
    'unit': 'fL',
    'table': 'labevents',
    'code': '(51250//MCV)',
    'ids': [51250],
    'src_regex': None,
    'short': 'mcv',
    'min': 50,
    'max': 150}],
  'white_blood_cell_count': [{'name': 'white_blood_cell_count',
    'unit': ['K/uL', 'G/l'],
    'table': 'labevents',
    'code': '(51301//White Blood Cells)',
    'ids': [51301],
    'src_regex': None,
    'short': 'wbc',
    'min': 0,
    'max': 'Inf'}],
  'hematocrit': [{'name': 'hematocrit',
    'unit': '%',
    'table': 'labevents',
    'code': '(51221//Hematocrit)',
    'ids': [51221],
    'src_regex': None,
    'short': 'hct',
    'min': 15,
    'max': 60}],
  'partial_thromboplastin_time': [{'name': 'partial_thromboplastin_tim

In [6]:
data_by_cnpt = {}
for cat in data_by_cat.keys():
    for cnpt in data_by_cat[cat]:
        data_by_cnpt[cnpt] = data_by_cat[cat][cnpt]

## Overview

In [7]:
# Overview about usable data
print("total category count:", len(data_by_cat))
for key in data_by_cat.keys():
    print("\tcategory:", key, "concept count:", len(data_by_cat[key]))
print("total concept count:", len(data_by_cnpt))

total category count: 9
	category: hematology concept count: 20
	category: neurological concept count: 4
	category: blood_gas concept count: 9
	category: output concept count: 1
	category: chemistry concept count: 21
	category: medications concept count: 13
	category: vitals concept count: 6
	category: respiratory concept count: 4
	category: demographics concept count: 2
total concept count: 80


In [8]:
# Overview about unused data
print("total unused concept count:", len(un_src_data))


total unused concept count: 8


## Saves data 
Saves as data.json for manual inspections

In [9]:
with open("data.json", "w") as file:
    json.dump(data_by_cat, file, indent=4, ensure_ascii=False)

In [10]:
len(un_src_data)

8

In [11]:
# with open("non-finished-mimic-concepts.json", "w") as file:
#     json.dump(un_src_data, file, indent=4, ensure_ascii=False)

In [12]:
# for key, value in data_by_cat.items():
#     print(value.keys())

## Function
creates config.ymls

In [13]:
def create(dir_path: str, name: str, unit: str, table_contents: list[dict]) -> None:
    if unit == "%":
        unit = "\"%\""

    path = Path(dir_path)
    path.mkdir(parents=True, exist_ok=True)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = f"""name: {name}
version: 1.0.0
unit: {unit}

extension_columns:
  dataset: col("dataset")
  table: col("table")

mappings:
"""
    for table_content in table_contents:
      content +=  f"""  - pattern:
      dataset: mimic-iv
      version: "3.1"
      table: {table_content["table"]}
      event: result
      code: {table_content["code"]}
    columns:
      numeric_value: col(numeric_value)
      text_value: col(text_value)
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

## Execution
TODO: make pattern if unit ist list

In [14]:
base_path = Path("/workspaces/config/concept")

for category in data_by_cat.keys():
    folder = base_path / category.replace(" ", "_").replace("/", "_per_")
    folder.mkdir(parents=True, exist_ok=True)
    for cnpt in (category := data_by_cat[category]).keys() :
        create(folder,
        category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))]
        )
    

File already exists: /workspaces/config/concept/hematology/basophils.yml
File already exists: /workspaces/config/concept/hematology/mean_corpuscular_volume.yml
File already exists: /workspaces/config/concept/hematology/white_blood_cell_count.yml
File already exists: /workspaces/config/concept/hematology/hematocrit.yml
File already exists: /workspaces/config/concept/hematology/partial_thromboplastin_time.yml
File already exists: /workspaces/config/concept/hematology/red_blood_cell_count.yml
File already exists: /workspaces/config/concept/hematology/lymphocytes.yml
File already exists: /workspaces/config/concept/hematology/prothrombine_time.yml
File already exists: /workspaces/config/concept/hematology/hemoglobin.yml
File already exists: /workspaces/config/concept/hematology/platelet_count.yml
File already exists: /workspaces/config/concept/hematology/mean_corpuscular_hemoglobin_concentration.yml
File already exists: /workspaces/config/concept/hematology/mean_corpuscular_hemoglobin.yml
F

# Run pipeline.ipynb

In [15]:
# Run pipeline.ipynb

# Sanity check

In [16]:
# Prepare needed information to make check inside of ricu/ R

# Root folder were the concepts are saved
cnpt_root_folder = Path("/workspaces/OpenICU-Example/output/project/workspace/concept")

found_data_info = {}

# iterate all the folders in the root folder
for cnpt_folder in cnpt_root_folder.iterdir():
    # if the cnpt_folder is a folder/directory execute logic
    if cnpt_folder.is_dir():
        # select the first file in the sub folder "/1.0.0/" and exprect it to be the cnpt_file
        cnpt_file  = next((file for file in (cnpt_folder / "1.0.0").iterdir() if file.is_file()), None)
        # print(f"{cnpt_folder}: {cnpt_file.name if cnpt_file else "empty"}")
        size = len(pl.read_parquet(cnpt_file))
        # print(size)
        name = cnpt_folder.name
        if data_by_cnpt.get(name):
            found_data_info[name] = data_by_cnpt[name]
        else:
            print("Not found: ", name)

In [17]:
with open("check-open_icu-to-ricu.json", "w") as file:
    json.dump(found_data_info, file, indent=4, ensure_ascii=False)
# found_data_info

## Make Checklist for Github

In [18]:

def make_issue_checklist(ricu: pl.DataFrame):
    cat = {}

    for col in ricu.columns:
        elem = list(ricu[col])[0]
        if not cat.get(elem["category"]):
            cat[elem["category"]] = []
        cat[elem["category"]] += [elem["description"]]
    output = ""
    for cat, cnpts in cat.items():
        output += f"- [ ] {cat}:\n"
        for cnpt in cnpts:
            output += ((f"  - [x] {cnpt}\n") if (cnpt.replace(" ", "_").replace("/", "_per_") in data_by_cnpt.keys()) else (f"  - [ ] {cnpt}\n"))
            if (cnpt.replace(" ", "_").replace("/", "_per_") not in data_by_cnpt.keys()):
                print(f"{cnpt}")
    return output

In [19]:
output = make_issue_checklist(ricu_cnpt_dict_df)

GCS total
AVPU scale
Glasgow coma scale (non-sedated)
carboxyhemoglobin
national early warning score
sepsis-3 criterion
hospital length of stay
in hospital mortality
ICU length of stay
SOFA cardiovascular component
SOFA coagulation component
SOFA liver component
suspected infection
systemic inflammatory response syndrome score
dopamine administration for min 1h
norepinephrine administration for min 1h
quick SOFA score
SOFA renal component
SOFA central nervous system component
SOFA respiratory component
modified early warning score
epinephrine administration for min 1h
sequential organ failure assessment score
urine output per 24h
norepinephrine equivalents
vasopressor indicator
dobutamine administration for min 1h
corticosteroids
patient sex
patient admission type
patient age
patient body mass index
ventilation end
ventilation start
supplemental oxygen
tracheostomy
SaO2/FiO2
Horowitz index
ventilation durations
body fluid sampling
